# Antes de implementar RAG
Ejemplo de como responde el modelo antes de implementar la arquitectura RAG

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Inicializar el modelo
model = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash-lite",
    temperature = 0,
)

# Le decimos al modelo lo que queremos que haga
messages = [
    (
        "system",
        """Eres un asesor financiero que responde preguntas de los usuarios con explicaciones
        adecuadas y concisas para quienes no estan acostumbrados al lenguaje de las finanzas. 
        Tus respuestas deben estar en el idioma en el que fue hecha la pregunta""",
    ),
    ("human", "Segun Aharon L. Lechter que consejo malo dan los padres a sus hijos hoy en dia?"),
]

# Imprimir la respuesta del modelo
msg = model.invoke(messages)
print(msg.content)

Según Aharon L. Lechter, un consejo financiero que los padres suelen dar a sus hijos y que puede ser contraproducente hoy en día es:

**"Ahorra todo lo que puedas y no gastes en cosas que no necesitas."**

**¿Por qué puede ser un mal consejo hoy en día?**

Si bien el ahorro es importante, este consejo, dicho de forma muy estricta, puede llevar a que los hijos:

*   **Se pierdan oportunidades de crecimiento:** Invertir en educación, en un negocio propio, o incluso en experiencias que amplíen sus horizontes, puede ser más beneficioso a largo plazo que simplemente acumular dinero sin usarlo.
*   **Tengan miedo a gastar:** Pueden desarrollar una mentalidad de escasez, donde gastar dinero, incluso en cosas que les traerían felicidad o mejorarían su calidad de vida, se sienta como un error.
*   **No aprendan a usar el dinero de forma inteligente:** Ahorrar es solo una parte de la salud financiera. Aprender a invertir, a gastar de forma consciente y a gestionar deudas (si es necesario) son ha

# Implementacion RAG

### Document loading

In [5]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

path = Path("./books")
allDocs = []
# Iteramos cada archivo en la ruta
for file in path.iterdir():
    loader = PyPDFLoader(file)
    pages = loader.load()
    # Interceptamos el archivo para modificar su metadata
    for page in pages:
        if file.name == "Padre_Pobre_Padre_Rico.pdf":
            page.metadata['title'] = 'Padre Rico, Padre Pobre'
            page.metadata['author'] = 'Robert T. Kiyosaki, Sharon L. Lechter' 
            page.metadata['subject'] = 'Finanzas Personales'
    # print(pages[0].metadata)
    allDocs.extend(pages)
    print(f"File {file.name} loaded. Pages: {len(pages)}")

File EL INVERSOR INTELIGENTE - BENJAMIN GRAHAM.pdf loaded. Pages: 1044
File el-pequeno-libro-para-invertir-john-c-bogle.pdf loaded. Pages: 165
File El_hombre_mas_rico_de_Babilonia_George_S_Clason.pdf loaded. Pages: 121
File Los secretos de la mente millonaria - T. Harv Eker.pdf loaded. Pages: 140
File Padre_Pobre_Padre_Rico.pdf loaded. Pages: 231
File Pequeño Cerdo Capitalista.pdf loaded. Pages: 150
File The-Psychology-of-Money-Morgan-Housel.pdf loaded. Pages: 242


### Document splitting

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunkSize = 1200
chunkOverlap = 200

rSplitter = RecursiveCharacterTextSplitter(
    chunk_size = chunkSize,
    chunk_overlap = chunkOverlap
)

splittedDocs = rSplitter.split_documents(allDocs)
print(f"Páginas originales: {len(allDocs)}")
print(f"Trozos (chunks) generados listos para la base de datos: {len(splittedDocs)}")

Páginas originales: 2093
Trozos (chunks) generados listos para la base de datos: 4192
